# Data Cleaning — Online Retail Dataset
**Milestone: Data Cleaning** | Python / Google Colab

## Objective
Prepare a trustworthy dataset for positive-value retail sales analysis using genuine transaction records, while preserving an audit trail. The raw data contains 541,909 records and eight columns. It covers a UK online retailer from December 2010 to December 2011.

**Source:** Daqing Chen (2015), [Online Retail, UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/352/online+retail), DOI: 10.24432/C5BW33. No artificial errors are introduced.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx'
raw = pd.read_excel(url, engine='openpyxl')
df = raw.copy()
print('Raw shape:', raw.shape)
display(raw.sample(10, random_state=42))

## 1. Initial quality assessment
Inspect data types, missing values, exact duplicates, numeric distributions and text variants. IDs are labels, not measurements.

In [ ]:
raw.info()
display(pd.DataFrame({'Missing': raw.isna().sum(), 'Percent': raw.isna().mean()*100}))
print('Exact duplicate rows:', raw.duplicated().sum())
display(raw[['Quantity','UnitPrice']].describe())
for col in ['Description','Country']:
    s = raw[col].astype('string')
    print(col, 'whitespace variants:', int((s != s.str.strip()).sum()),
          'distinct before/after normalization:', s.nunique(), s.str.strip().str.upper().nunique())

## 2. Duplicate handling and formatting
Remove exact duplicate rows, keeping the first. Since there is no invoice-line identifier, some repeats may be legitimate; this is a documented assumption. Do not deduplicate on invoice number alone because orders contain multiple products. Standardize case and whitespace in descriptions, stock codes, invoice numbers and countries. Keep identifiers as strings and parse dates. Remove additional exact duplicates created by normalization under the same assumption.

In [ ]:
audit = []
def record(step, before, after):
    audit.append({'Step':step,'Before':before,'After':after,'Removed':before-after})
before = len(df)
df = df.drop_duplicates().copy()
record('Exact duplicates', before, len(df))
for col in ['InvoiceNo','StockCode','Description','Country']:
    df[col] = df[col].astype('string').str.strip().str.upper().replace('', pd.NA)
df['CustomerID'] = pd.to_numeric(df['CustomerID'], errors='coerce').astype('Int64').astype('string')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')
for col in ['Quantity','UnitPrice']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
before = len(df)
df = df.drop_duplicates().copy()
record('Normalized duplicates', before, len(df))
display(df.isna().sum())

## 3. Missing values: column-specific decisions
| Column | Strategy and reason |
|---|---|
| InvoiceNo | Drop missing rows: transaction cannot be identified. |
| StockCode | Drop missing rows: product cannot be identified. |
| Description | Use most frequent known description for the same stock code; otherwise UNKNOWN PRODUCT. Never use an unrelated global mode. |
| Quantity | Drop missing rows: inventing quantities changes sales totals. |
| InvoiceDate | Drop missing rows: required for time analysis. |
| UnitPrice | Drop missing rows: invented prices distort revenue. |
| CustomerID | Use UNKNOWN and retain a missing-ID indicator. Do not treat UNKNOWN as a single actual customer or use it for customer-level analysis. |
| Country | Use UNKNOWN if missing; assigning the most frequent country would bias geography. |

Rules for columns with no missing values are defensive and may remove zero rows.

In [ ]:
before = len(df)
df = df.dropna(subset=['InvoiceNo','StockCode','Quantity','InvoiceDate','UnitPrice']).copy()
record('Missing essential fields', before, len(df))
df['DescriptionWasMissing'] = df['Description'].isna().astype('int8')
known = df.dropna(subset=['Description'])
counts = known.groupby(['StockCode','Description']).size().rename('Frequency').reset_index()
lookup = counts.sort_values(['StockCode','Frequency','Description'], ascending=[True,False,True]).drop_duplicates('StockCode').set_index('StockCode')['Description']
df['Description'] = df['Description'].fillna(df['StockCode'].map(lookup)).fillna('UNKNOWN PRODUCT')
df['CustomerIDWasMissing'] = df['CustomerID'].isna().astype('int8')
df['CustomerID'] = df['CustomerID'].fillna('UNKNOWN')
df['Country'] = df['Country'].fillna('UNKNOWN')
display(df.isna().sum())

## 4. Outlier detection and decisions
Use the 1.5 × IQR rule for Quantity and UnitPrice. Values below Q1 − 1.5 × IQR or above Q3 + 1.5 × IQR are flagged, not automatically deleted. Wholesale orders and expensive products can legitimately be extreme. Negative quantities and cancellation invoices are excluded from the positive-sales dataset, not described as errors. Nonpositive prices and quantities are excluded because the objective is positive-value sales; they may represent adjustments or free items. Save a row-level audit of every excluded or IQR-flagged record with its decision. The source workbook remains unchanged.

In [ ]:
numeric = ['Quantity','UnitPrice']
fig, axes = plt.subplots(1,2,figsize=(10,4))
for col, ax in zip(numeric,axes):
    ax.boxplot(df[col].dropna())
    ax.set_title(col + ' before filtering')
plt.tight_layout(); plt.show()
limits = []
for col in numeric:
    q1,q3 = df[col].quantile([.25,.75])
    iqr = q3-q1
    low,high = q1-1.5*iqr,q3+1.5*iqr
    flag = df[col].lt(low) | df[col].gt(high)
    df[col+'Outlier'] = flag.astype('int8')
    limits.append({'Column':col,'Q1':q1,'Q3':q3,'IQR':iqr,'Lower':low,'Upper':high,'Flagged':int(flag.sum())})
display(pd.DataFrame(limits))
cancel = df['InvoiceNo'].str.startswith('C')
nonqty = df['Quantity'].le(0)
nonprice = df['UnitPrice'].le(0)
exclude = cancel | nonqty | nonprice
flagged = df['QuantityOutlier'].eq(1) | df['UnitPriceOutlier'].eq(1)
review = df.loc[exclude | flagged].copy()
review['Cancellation'] = cancel.loc[review.index]
review['NonpositiveQuantity'] = nonqty.loc[review.index]
review['NonpositivePrice'] = nonprice.loc[review.index]
review['Decision'] = np.where(exclude.loc[review.index], 'Exclude from positive sales', 'Retain: plausible extreme; not verified as error')
display(review.groupby(['Decision','Cancellation','NonpositiveQuantity','NonpositivePrice']).size().rename('Rows').reset_index())
display(review.head(10))
before = len(df)
df = df.loc[~exclude].copy()
record('Outside positive-sales scope', before, len(df))
# Keep original extreme values and flags; do not invent replacement prices.
df['TotalAmount'] = df['Quantity'] * df['UnitPrice']

## 5. Categorical encoding
Country is nominal and has no meaningful order: use one-hot encoding. InvoiceNo, StockCode and CustomerID are high-cardinality identifiers; keep them as metadata rather than assigning artificial numeric ranks. Description is free text and is retained as metadata. InvoiceDate is converted into numeric calendar features. Missingness and outlier indicators are binary. No genuinely ordinal categories exist, so ordinal encoding is not appropriate. For future predictive modeling, learn imputations, IQR thresholds and encoder categories on training data only to avoid leakage.

In [ ]:
clean = df.reset_index(drop=True)
features = clean[['Quantity','UnitPrice','DescriptionWasMissing','CustomerIDWasMissing','QuantityOutlier','UnitPriceOutlier']].copy()
features['Month'] = clean['InvoiceDate'].dt.month
features['DayOfWeek'] = clean['InvoiceDate'].dt.dayofweek
features['Hour'] = clean['InvoiceDate'].dt.hour
country_encoded = pd.get_dummies(clean['Country'], prefix='Country', dtype='int8')
features = pd.concat([features,country_encoded],axis=1)
print('Encoded feature shape:',features.shape)
display(features.head())

## 6. Before / after comparison
Compare original and cleaned transaction tables on the same columns, separately from the encoded feature table. Changes in numeric statistics reflect both duplicate handling and the positive-sales scope, not necessarily correction of erroneous values.

In [ ]:
comparison = pd.DataFrame({'Before':[raw.shape[0],raw.shape[1],raw.isna().sum().sum(),raw.duplicated().sum()], 'After':[clean.shape[0],clean.shape[1],clean.isna().sum().sum(),clean[raw.columns].duplicated().sum()]},index=['Rows','Columns','Missing cells','Exact duplicates (original fields)'])
display(comparison)
display(pd.DataFrame(audit))
missing = pd.DataFrame({'Before':raw.isna().sum(),'After':clean[raw.columns].isna().sum()})
display(missing)
stats = pd.concat({'Before':raw[numeric].describe(),'After':clean[numeric].describe()},axis=1)
display(stats)
missing.plot.bar(figsize=(10,4),title='Missing values before and after cleaning')
plt.ylabel('Missing cells'); plt.tight_layout(); plt.show()
fig, axes = plt.subplots(1,2,figsize=(10,4))
for col,ax in zip(numeric,axes):
    ax.boxplot(clean[col]); ax.set_title(col+' after filtering')
plt.tight_layout(); plt.show()

## 7. Validation and export
Check that the cleaned table contains no missing cells, all retained sales have positive quantities and prices, and the encoded feature matrix is numeric. Imputation can make originally distinct rows identical on the original fields; preserve provenance flags rather than removing these as proven duplicates.

In [ ]:
assert clean.isna().sum().sum() == 0
assert (clean['Quantity'] > 0).all() and (clean['UnitPrice'] > 0).all()
assert not clean['InvoiceNo'].str.startswith('C').any()
assert features.isna().sum().sum() == 0
assert all(pd.api.types.is_numeric_dtype(t) for t in features.dtypes)
assert len(raw)-len(clean) == sum(x['Removed'] for x in audit)
assert (country_encoded.sum(axis=1)==1).all()
print('Validation passed.')
out = Path('outputs'); out.mkdir(exist_ok=True)
clean.to_csv(out/'cleaned_online_retail.csv',index=False)
features.to_csv(out/'encoded_features.csv',index=False)
review.to_csv(out/'outlier_decisions.csv',index_label='OriginalRowIndex')
pd.DataFrame(audit).to_csv(out/'cleaning_audit.csv',index=False)
comparison.to_csv(out/'before_after.csv')
missing.to_csv(out/'missing_comparison.csv')
stats.to_csv(out/'summary_statistics.csv')
print('Saved files:', [x.name for x in out.iterdir()])

## Conclusion
The workflow addresses duplicates, text inconsistencies, missing values, extreme observations and categorical encoding using documented, reproducible rules. Unknown customers and plausible extremes are preserved where appropriate. The result is suitable for positive-sales analysis, but not for net revenue or returns analysis. Review the actual output counts and audit tables before submission.

### References
- [UCI Online Retail dataset](https://archive.ics.uci.edu/dataset/352/online+retail)
- [Pandas missing data guide](https://pandas.pydata.org/docs/user_guide/missing_data.html)
- [Scikit-learn preprocessing](https://scikit-learn.org/stable/modules/preprocessing.html)